In [ ]:
# CELL 1 - settings and imports
import os, json, time
import numpy as np
CAP = 1500            # a concept list longer than this is stopped; that run is 'intractable' and its CDI is excluded
TH_U, TH_D = 0.60, 0.30
OUT = "results"
os.makedirs(OUT, exist_ok=True)
print("ready")

In [ ]:
# CELL 2 - data generator (Table X)
def make_context(rng, n_obj, assign, mem, n_inf, n_amb, n_uni, noise):
    """Return T, I, F arrays of shape (n_obj, n_inf+n_amb+n_uni). Informative columns come first."""
    u = rng.uniform
    member = np.array([[assign[g] in mem[j] for j in range(n_inf)] for g in range(n_obj)])
    sh = (n_obj, n_inf)
    T = np.where(member, u(.65, 1, sh), u(0, .35, sh))
    I = u(0, .25, sh)
    F = np.where(member, u(0, .35, sh), u(.65, 1, sh))
    noisy = rng.random(sh) < noise                       # informative cell replaced by an ambiguous triple
    T = np.where(noisy, u(0, 1, sh), T); F = np.where(noisy, u(0, 1, sh), F); I = np.where(noisy, u(.4, 1, sh), I)
    sa = (n_obj, n_amb)
    Ta, Fa, Ia = u(0, 1, sa), u(0, 1, sa), u(.4, 1, sa)            # ambiguous attributes
    su = (n_obj, n_uni)
    Tu, Iu, Fu = u(0, .25, su), u(0, .25, su), u(.75, 1, su)       # uninformative attributes (false for everyone)
    return (np.hstack([T, Ta, Tu]), np.hstack([I, Ia, Iu]), np.hstack([F, Fa, Fu]))

GEN1 = dict(n_obj=40, n_clusters=4, n_inf=8, n_amb=3, n_uni=3, max_k=2)
GEN2 = {20: dict(n_obj=60, n_clusters=6, n_inf=8, n_amb=6, n_uni=6, max_k=3),
        40: dict(n_obj=60, n_clusters=6, n_inf=16, n_amb=12, n_uni=12, max_k=3),
        80: dict(n_obj=60, n_clusters=6, n_inf=32, n_amb=24, n_uni=24, max_k=3)}

def make_pair(seed, delta, gen, noise=0.10, shift=0.0):
    """Two contexts. Same seed -> same t1 for every delta. delta = share of objects moved to another cluster at t2."""
    rng = np.random.default_rng(seed)
    n, nc = gen["n_obj"], gen["n_clusters"]
    assign = np.repeat(np.arange(nc), n // nc)
    mem = [set(rng.choice(nc, size=rng.integers(1, gen["max_k"] + 1), replace=False)) for _ in range(gen["n_inf"])]
    args = (gen["n_inf"], gen["n_amb"], gen["n_uni"], noise)
    c1 = make_context(rng, n, assign, mem, *args)
    a2 = assign.copy()
    for g in rng.choice(n, size=int(round(delta * n)), replace=False):
        a2[g] = rng.choice([c for c in range(nc) if c != assign[g]])
    c2 = make_context(rng, n, a2, mem, *args)
    if shift:
        c2 = (np.minimum(c2[0] + shift, 1.0), c2[1], c2[2])          # calibration shift of all truth values, clipped at 1
    return c1, c2


In [ ]:
# CELL 3 - ambiguity U, discrimination D, attribute screening (Eqs. 3-6)
def scores(T, I, F):
    d = np.abs(T - F) * (1 - I)
    return 1 - d.mean(0), (T - F).std(0)

def shannon(T, I, F):
    P = np.stack([T, I, F], -1); P = P / np.maximum(P.sum(-1, keepdims=True), 1e-12)
    return (-(np.where(P > 0, P * np.log(np.where(P > 0, P, 1)), 0)).sum(-1) / np.log(3)).mean(0)

def select(c1, c2, how, th_u=TH_U, th_d=TH_D):
    m = c1[0].shape[1]
    if how == "all": return np.arange(m)
    if how == "shannon":
        return np.where((shannon(*c1) <= 0.60) & (shannon(*c2) <= 0.60))[0]
    U1, D1 = scores(*c1); U2, D2 = scores(*c2)
    okU = (U1 <= th_u) & (U2 <= th_u); okD = (D1 >= th_d) & (D2 >= th_d)
    ok = {"UD": okU & okD, "U": okU, "D": okD}[how]
    return np.where(ok)[0]


In [ ]:
# CELL 4 - neutrosophic to binary table (Eqs. 7-8) and the baseline rules
def binarise(c, keep, thr):
    T, I, F = c
    if len(keep) == 0: return np.zeros((T.shape[0], 0), dtype=np.uint8)
    if thr == "mean_all":   a, b, g = T.mean(), I.mean(), F.mean()
    elif thr == "mean_Mp":  a, b, g = T[:, keep].mean(), I[:, keep].mean(), F[:, keep].mean()
    elif thr == "median_all": a, b, g = np.median(T), np.median(I), np.median(F)
    elif thr == "fixed":    a, b, g = .5, .5, .5
    elif thr == "crisp":    return (T[:, keep] >= .5).astype(np.uint8)
    return ((T[:, keep] >= a) & (I[:, keep] <= b) & (F[:, keep] <= g)).astype(np.uint8)


In [ ]:
# CELL 5 - concepts (NextClosure) and the Concept Drift Index (Eqs. 9-11)
def concepts(B, cap=CAP):
    """All formal concepts as (extent_bits, intent_bits). Returns (list, capped)."""
    n, m = B.shape
    full = (1 << n) - 1
    if m == 0: return [(full, 0)], False
    cols = [sum(1 << int(g) for g in np.nonzero(B[:, j])[0]) for j in range(m)]
    def closure(a):
        ext = full; x = a
        while x:
            low = x & -x; ext &= cols[low.bit_length() - 1]; x ^= low
        inten = 0
        for j in range(m):
            if cols[j] & ext == ext: inten |= 1 << j
        return ext, inten
    ext, A = closure(0); out = [(ext, A)]; allm = (1 << m) - 1
    while A != allm:
        nxt = None
        for i in range(m - 1, -1, -1):
            if (A >> i) & 1: continue
            mask = (1 << i) - 1
            e2, B2 = closure((A & mask) | (1 << i))
            if (B2 & mask) == (A & mask): nxt = (e2, B2); break
        if nxt is None: break
        out.append(nxt); A = nxt[1]
        if len(out) >= cap: return out, True
    return out, False

def _j(a, b):
    u = (a | b).bit_count()
    return 1.0 if u == 0 else (a & b).bit_count() / u

def cdi(C1, C2):
    S = np.array([[0.5 * (_j(c[0], d[0]) + _j(c[1], d[1])) for d in C2] for c in C1])
    return 1 - 0.5 * (S.max(1).mean() + S.max(0).mean())


In [ ]:
# CELL 6 - the 11 methods compared
METHODS = {   # name: (selection, threshold rule)
 "mscope":        ("UD", "mean_Mp"),
 "global":        ("UD", "mean_all"),
 "median":        ("UD", "median_all"),
 "U_only":        ("U",  "mean_all"),
 "D_only":        ("D",  "mean_all"),
 "shannon":       ("shannon", "mean_all"),
 "noreduc_mean":  ("all", "mean_all"),
 "noreduc_fixed": ("all", "fixed"),
 "fixed_scr":     ("UD", "fixed"),
 "crisp":         ("all", "crisp"),
 "crisp_scr":     ("UD", "crisp"),
}

def run_method(c1, c2, name, n_inf, th_u=TH_U, th_d=TH_D, need_conc=True):
    sel, thr = METHODS[name]
    keep = select(c1, c2, sel, th_u, th_d)
    B1, B2 = binarise(c1, keep, thr), binarise(c2, keep, thr)
    r = dict(k=int(len(keep)), tp=int((keep < n_inf).sum()),
             ham=float((B1 != B2).mean()) if len(keep) else 0.0)
    if need_conc:
        t0 = time.time(); C1, cap1 = concepts(B1); C2, cap2 = concepts(B2); r["sec"] = time.time() - t0
        r["capped"] = bool(cap1 or cap2); r["nc"] = (len(C1) + len(C2)) / 2
        r["cdi"] = None if r["capped"] else float(cdi(C1, C2))
        # share of columns outside M' that hold a 1 at t1 (used for the 'removed columns were empty' claim)
        out = np.setdiff1d(np.arange(c1[0].shape[1]), keep)
        r["out_ones"] = int(binarise(c1, out, thr).sum()) if len(out) else 0
    return r

def raw_dist(c1, c2):
    return float(np.sqrt((c1[0] - c2[0]) ** 2 + (c1[1] - c2[1]) ** 2 + (c1[2] - c2[2]) ** 2).mean())


In [ ]:
# CELL 7 - the five studies (each function runs one seed)
def task_round1(a):
    seed, = a; out = []
    for d in (0, .05, .10, .20, .30, .40):
        c1, c2 = make_pair(seed, d, GEN1)
        rec = dict(seed=seed, delta=d, raw=raw_dist(c1, c2))
        for m in ("mscope", "global", "U_only", "D_only", "shannon", "noreduc_mean", "noreduc_fixed", "crisp"):
            rec[m] = run_method(c1, c2, m, 8)
        out.append(rec)
    return out

GRID_U = (.3, .4, .5, .6, .7, .8, .9); GRID_D = (.1, .2, .3, .4, .5, .6)
def task_sens(a):
    seed, = a; rec = []
    for d in (0, .2):
        c1, c2 = make_pair(seed, d, GEN1)
        for tu in GRID_U:
            for td in GRID_D:
                for m in ("mscope", "global"):
                    r = run_method(c1, c2, m, 8, tu, td); rec.append(dict(seed=seed, delta=d, tu=tu, td=td, method=m, k=r["k"], cdi=r["cdi"]))
    return rec

def task_noise(a):
    seed, noise = a; rec = []
    for d in (0, .2):
        c1, c2 = make_pair(seed, d, GEN1, noise=noise)
        for m in ("mscope", "global", "noreduc_mean", "noreduc_fixed", "crisp"):
            r = run_method(c1, c2, m, 8); rec.append(dict(seed=seed, noise=noise, delta=d, method=m, k=r["k"], cdi=r["cdi"], capped=r["capped"]))
    return rec

def task_scale(a):
    seed, n_attr = a; rec = []; gen = GEN2[n_attr]
    for d in (0, .2):
        c1, c2 = make_pair(seed, d, gen)
        raw = raw_dist(c1, c2)
        for m in ("crisp", "crisp_scr", "noreduc_fixed", "fixed_scr", "noreduc_mean", "global", "median"):
            r = run_method(c1, c2, m, gen["n_inf"])
            rec.append(dict(seed=seed, n_attr=n_attr, delta=d, method=m, raw=raw, **{k: r[k] for k in ("k", "nc", "sec", "capped", "cdi", "ham", "tp")}))
    return rec

def task_calib(a):
    seed, = a; rec = []; gen = GEN2[20]
    scen = [("none", 0.0, 0.0), ("shift0.10", 0.0, 0.10), ("shift0.20", 0.0, 0.20), ("structural", 0.2, 0.0)]
    for name, d, s in scen:
        c1, c2 = make_pair(seed, d, gen, shift=s)
        r = dict(seed=seed, scen=name, raw=raw_dist(c1, c2))
        for m in ("global", "mscope", "noreduc_fixed", "crisp"):
            q = run_method(c1, c2, m, gen["n_inf"]); r[m] = q["cdi"]; r[m + "_ham"] = q["ham"]; r[m + "_k"] = q["k"]
        rec.append(r)
    return rec


In [ ]:
# CELL 8 - run everything (about 1-3 minutes; no multiprocessing, so it works in Jupyter on Windows)
def run(name, fn, tasks):
    t0 = time.time()
    flat = [x for t in tasks for x in fn(t)]
    json.dump(flat, open(f"{OUT}/{name}.json", "w"))
    print(f"{name}: {len(flat)} records, {time.time()-t0:.0f}s")
    return flat

R1    = run("round1", task_round1, [(1000 + s,) for s in range(30)])
SENS  = run("sens",   task_sens,   [(2000 + s,) for s in range(20)])
NOISE = run("noise",  task_noise,  [(3000 + s, z) for z in (0, .1, .2, .3) for s in range(20)])
CALIB = run("calib",  task_calib,  [(7000 + s,) for s in range(20)])
SCALE = run("scale",  task_scale,  [(5000 + s, na) for na in (20, 40, 80) for s in range(12)])

In [ ]:
# CELL 9 - summary: send me this printed output
def auc(a, b):
    return float(np.mean([(x > y) + .5 * (x == y) for x in a for y in b])) if a and b else float("nan")
def col(rows, m, d):
    return [x[m]["cdi"] for x in rows if x["delta"] == d and x[m]["cdi"] is not None]
print("ROUND 1: mean CDI at delta = 0, .05, .10, .20, .30, .40  |  AUC(.2 vs 0)  |  attributes kept")
for m in ("mscope", "global", "U_only", "D_only", "shannon", "noreduc_mean", "noreduc_fixed", "crisp"):
    row = [np.mean(col(R1, m, d)) for d in (0, .05, .10, .20, .30, .40)]
    k = np.mean([x[m]["k"] for x in R1 if x["delta"] == 0])
    print(f"{m:14s}", " ".join(f"{v:.3f}" for v in row), f"| AUC {auc(col(R1, m, .2), col(R1, m, 0)):.2f} | k {k:.1f}")
print("\nNOISE: false drift / AUC")
for m in ("mscope", "global", "noreduc_mean", "noreduc_fixed", "crisp"):
    s = []
    for z in (0, .1, .2, .3):
        a = [x["cdi"] for x in NOISE if x["method"] == m and x["noise"] == z and x["delta"] == 0 and x["cdi"] is not None]
        b = [x["cdi"] for x in NOISE if x["method"] == m and x["noise"] == z and x["delta"] == .2 and x["cdi"] is not None]
        s.append(f"{np.mean(a):.3f}/{auc(b, a):.2f}")
    print(f"{m:14s}", " ".join(s))
print("\nSCALE: capped runs / mean concepts / false drift / AUC")
for na in (20, 40, 80):
    for m in ("crisp", "noreduc_fixed", "noreduc_mean", "global", "median"):
        q = [x for x in SCALE if x["n_attr"] == na and x["method"] == m]
        a = [x["cdi"] for x in q if x["delta"] == 0 and x["cdi"] is not None]
        b = [x["cdi"] for x in q if x["delta"] == .2 and x["cdi"] is not None]
        print(na, f"{m:14s}", sum(x["capped"] for x in q), "/", len(q), f"nc {np.mean([x['nc'] for x in q]):.0f}",
              f"fd {np.mean(a):.3f} AUC {auc(b, a):.2f}" if a and b else "fd n/a (intractable)")
print("\nCALIBRATION SHIFT: mean CDI")
for m in ("global", "mscope", "noreduc_fixed", "crisp"):
    print(f"{m:14s}", {s: round(float(np.mean([x[m] for x in CALIB if x["scen"] == s and x[m] is not None])), 3) for s in ("none", "shift0.10", "shift0.20", "structural")})

In [ ]:
# CELL 10 - helper functions for the tables (Spearman, AUC, bootstrap)
import numpy as np, json
def auc(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    if len(a) == 0 or len(b) == 0: return float("nan")
    return float(((a[:, None] > b[None, :]) + .5 * (a[:, None] == b[None, :])).mean())
def boot_auc(a, b, n=2000, seed=0):
    rng = np.random.default_rng(seed); a = np.asarray(a); b = np.asarray(b)
    v = [auc(rng.choice(a, len(a)), rng.choice(b, len(b))) for _ in range(n)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))
def rank(x):
    x = np.asarray(x, float); o = np.argsort(x, kind="mergesort"); r = np.empty(len(x)); r[o] = np.arange(len(x)); 
    for v in np.unique(x): r[x == v] = r[x == v].mean()
    return r
def spearman(x, y):
    rx, ry = rank(x), rank(y)
    if rx.std() == 0 or ry.std() == 0: return float("nan")
    return float(np.corrcoef(rx, ry)[0, 1])
S = {}   # every number used in the manuscript ends up in this dictionary
print("helpers ready")

In [ ]:
# CELL 11 - Tables XI-XVI and the numbers quoted in the text
R1  = json.load(open("results/round1.json"));  SENS  = json.load(open("results/sens.json"))
NOISE = json.load(open("results/noise.json")); SCALE = json.load(open("results/scale.json")); CALIB = json.load(open("results/calib.json"))
DELTAS = (0, .05, .10, .20, .30, .40)
# ---- Table XI
T11 = {}
for m in ("mscope", "global", "U_only", "D_only", "shannon", "noreduc_mean", "noreduc_fixed", "crisp"):
    runs = [x for x in R1 if x[m]["cdi"] is not None]
    k = np.mean([x[m]["k"] for x in R1]); tp = np.mean([x[m]["tp"] for x in R1])
    g = lambda d: [x[m]["cdi"] for x in R1 if x["delta"] == d]
    a, b = g(.2), g(0)
    T11[m] = dict(k=float(k), prec=float(tp / k) if k else 0.0, rec=float(tp / 8), nc=float(np.mean([x[m]["nc"] for x in R1])),
                  cdi0=(float(np.mean(g(0))), float(np.std(g(0), ddof=1))), cdi2=(float(np.mean(a)), float(np.std(a, ddof=1))), cdi4=(float(np.mean(g(.4))), float(np.std(g(.4), ddof=1))),
                  rho=spearman([x["delta"] for x in R1], [x[m]["cdi"] for x in R1]), auc=auc(a, b), ci=boot_auc(a, b),
                  curve=[(float(np.mean(g(d))), float(np.std(g(d), ddof=1))) for d in DELTAS])
S["T11"] = T11
d0 = [x for x in R1 if x["delta"] == 0]
S["shannon_empty_d0"] = int(sum(x["shannon"]["k"] == 0 for x in d0)); S["shannon_tp_d0"] = int(sum(x["shannon"]["tp"] for x in d0)); S["shannon_k_d0"] = float(np.mean([x["shannon"]["k"] for x in d0]))
S["screen_exact8"] = int(sum(x["global"]["k"] == 8 and x["global"]["tp"] == 8 for x in d0))
S["same_cdi_pairs"] = int(sum(abs(x["global"]["cdi"] - x["noreduc_mean"]["cdi"]) < 1e-12 for x in R1)); S["n_pairs"] = len(R1)
S["same_cdi_pairs_d_0_2_4"] = int(sum(abs(x["global"]["cdi"] - x["noreduc_mean"]["cdi"]) < 1e-12 for x in R1 if x["delta"] in (0, .2, .4)))
S["removed_ones_d0"] = int(sum(x["global"]["out_ones"] for x in d0))
S["crisp_nc"] = T11["crisp"]["nc"]; S["global_nc"] = T11["global"]["nc"]
# ---- mechanism behind the M'-scope result (re-generates the delta = 0 data sets)
Ip, Ia, cg, cm = [], [], [], []
for s in range(1000, 1030):
    c1, c2 = make_pair(s, 0, GEN1); keep = select(c1, c2, "UD"); T, I, F = c1
    Ip.append(I[:, keep].mean()); Ia.append(I.mean())
    clear = (T[:, :8] >= .65) & (I[:, :8] <= .25) & (F[:, :8] <= .35)
    for thr, store in (("mean_all", cg), ("mean_Mp", cm)):
        B = np.zeros(T.shape, int); B[:, keep] = binarise(c1, keep, thr)
        store.append(B[:, :8][clear].mean())
S["mech"] = dict(I_Mp=float(np.mean(Ip)), I_all=float(np.mean(Ia)), clear_global=float(np.mean(cg)), clear_mscope=float(np.mean(cm)))
# ---- Table XII (sensitivity)
T12 = {}
for tu in GRID_U:
    row = {}
    sel = [x for x in SENS if x["tu"] == tu and abs(x["td"] - .3) < 1e-9]
    row["k"] = float(np.mean([x["k"] for x in sel if x["method"] == "global" and x["delta"] == 0]))
    for m in ("mscope", "global"):
        a = [x["cdi"] for x in sel if x["method"] == m and x["delta"] == .2]; b = [x["cdi"] for x in sel if x["method"] == m and x["delta"] == 0]
        row[m] = auc(a, b)
    T12[str(tu)] = row
S["T12"] = T12
HEAT = {"k": [[float(np.mean([x["k"] for x in SENS if x["tu"] == tu and x["td"] == td and x["method"] == "global" and x["delta"] == 0])) for tu in GRID_U] for td in GRID_D]}
for m in ("mscope", "global"):
    HEAT[m] = [[auc([x["cdi"] for x in SENS if x["tu"] == tu and x["td"] == td and x["method"] == m and x["delta"] == .2],
                    [x["cdi"] for x in SENS if x["tu"] == tu and x["td"] == td and x["method"] == m and x["delta"] == 0]) for tu in GRID_U] for td in GRID_D]
S["HEAT"] = HEAT
S["plateau"] = [(tu, td) for i, td in enumerate(GRID_D) for j, tu in enumerate(GRID_U) if abs(HEAT["k"][i][j] - 8) < 1e-9]
# ---- Table XIII (noise)
T13 = {}
for m in ("mscope", "global", "noreduc_mean", "noreduc_fixed", "crisp"):
    T13[m] = {}
    for z in (0, .1, .2, .3):
        a = [x["cdi"] for x in NOISE if x["method"] == m and x["noise"] == z and x["delta"] == 0 and x["cdi"] is not None]
        b = [x["cdi"] for x in NOISE if x["method"] == m and x["noise"] == z and x["delta"] == .2 and x["cdi"] is not None]
        T13[m][str(z)] = (float(np.mean(a)), auc(b, a))
S["T13"] = T13
S["noise_capped"] = int(sum(x["capped"] for x in NOISE))
# ---- Table XIV (scale)
T14 = {}
for na in (20, 40, 80):
    for m in ("crisp", "crisp_scr", "noreduc_fixed", "fixed_scr", "noreduc_mean", "global"):
        q = [x for x in SCALE if x["n_attr"] == na and x["method"] == m]
        T14[f"{na}|{m}"] = dict(k=float(np.mean([x["k"] for x in q])), nc=float(np.mean([x["nc"] for x in q])), sec=float(np.mean([x["sec"] for x in q])), cap=int(sum(x["capped"] for x in q)), n=len(q))
S["T14"] = T14
# ---- Table XV (detection baselines) and false drift / median
T15, FD = {}, {}
for na in (20, 40, 80):
    def pair(m, key):
        q = [x for x in SCALE if x["n_attr"] == na and x["method"] == m]
        return [x[key] for x in q if x["delta"] == .2 and x[key] is not None], [x[key] for x in q if x["delta"] == 0 and x[key] is not None]
    T15[na] = dict(cdi=auc(*pair("global", "cdi")), cdi_fixed=auc(*pair("noreduc_fixed", "cdi")), ham=auc(*pair("global", "ham")), raw=auc(*pair("global", "raw")))
    FD[na] = dict(global_fd=float(np.mean(pair("global", "cdi")[1])), median_fd=float(np.mean(pair("median", "cdi")[1])) if pair("median", "cdi")[1] else None,
                  global_auc=T15[na]["cdi"], median_auc=auc(*pair("median", "cdi")), median_capped=int(sum(x["capped"] for x in SCALE if x["n_attr"] == na and x["method"] == "median")))
S["T15"] = T15; S["FD"] = FD
# ---- Table XVI (calibration shift)
scen = ("none", "shift0.10", "shift0.20", "structural")
T16 = {}
for name, key in (("global", "global"), ("mscope", "mscope"), ("fixed", "noreduc_fixed"), ("crisp", "crisp"), ("raw", "raw"), ("ham", "global_ham")):
    vals = {s: [x[key] for x in CALIB if x["scen"] == s and x[key] is not None] for s in scen}
    T16[name] = dict(means={s: float(np.mean(v)) for s, v in vals.items()}, p=auc(vals["structural"], vals["shift0.20"]))
S["T16"] = T16
json.dump(S, open(f"{OUT}/stats.json", "w"), indent=1, default=float)
print("tables computed -> results/stats.json")

In [ ]:
# CELL 12 - Figs 4-9 (saved as PNG and as vector PDF in the folder 'figs')
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 9, "axes.grid": False, "pdf.fonttype": 42})
os.makedirs("figs", exist_ok=True)
COL = dict(mscope="#c0392b", global_="#27ae60", noreduc_mean="#2980b9", noreduc_fixed="#8e44ad", crisp="#7f8c8d", shannon="black", raw="#2c3e50")
# which attributes does the Shannon filter keep? (0-7 informative, 8-10 ambiguous, 11-13 uninformative)
kept = [j for x in R1 if x["delta"] == 0 for j in select(*make_pair(x["seed"], 0, GEN1), "shannon")]
S["shannon_kept_uninformative_share"] = float(np.mean([j >= 11 for j in kept])) if kept else None
json.dump(S, open(f"{OUT}/stats.json", "w"), indent=1, default=float)

# Fig 4 - score plane of one data set at t1
c1, c2 = make_pair(1000, 0, GEN1); U, D = scores(*c1)
fig, ax = plt.subplots(figsize=(4.8, 3.5), dpi=200)
for sl, mk, c, lab in ((slice(0, 8), "o", "#27ae60", "Informative"), (slice(8, 11), "s", "#c0392b", "Ambiguous"), (slice(11, 14), "^", "#e67e22", "Uninformative")):
    ax.scatter(U[sl], D[sl], marker=mk, c=c, s=22, label=lab)
ax.axvline(TH_U, ls="--", c="gray"); ax.axhline(TH_D, ls="--", c="gray")
ax.set_xlabel("Ambiguity U"); ax.set_ylabel("Discriminability Δ"); ax.set_xlim(0.25, 0.95); ax.set_ylim(0, 0.75)
ax.text(0.31, 0.66, "kept", color="#27ae60", fontsize=8); ax.legend(fontsize=6.5, frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig("figs/fig4.png"); fig.savefig("figs/fig4.pdf"); plt.close(fig)

# Fig 5 - CDI against delta
fig, ax = plt.subplots(figsize=(6.4, 3.8), dpi=200)
spec = (("mscope", "M′-scope thresholds", "o", "-", COL["mscope"]), ("global", "Global thresholds (default)", "s", "-", COL["global_"]),
        ("noreduc_mean", "No reduction, mean thresholds", "^", "--", COL["noreduc_mean"]), ("noreduc_fixed", "No reduction, fixed (0.5,0.5,0.5)", "D", "-", COL["noreduc_fixed"]),
        ("crisp", "Crisp, T ≥ 0.5", "v", "-", COL["crisp"]), ("shannon", "Shannon filter", "x", ":", "black"))
for m, lab, mk, ls, c in spec:
    cu = np.array(S["T11"][m]["curve"]); ax.errorbar(DELTAS, cu[:, 0], yerr=cu[:, 1], label=lab, marker=mk, ls=ls, color=c, ms=4, lw=1.3, capsize=2, elinewidth=0.9)
ax.set_xlabel("Fraction of objects that changed cluster (δ)"); ax.set_ylabel("Mean CDI (± SD, 30 runs)"); ax.grid(alpha=.3)
ax.legend(fontsize=6.5, frameon=False, loc="lower right", bbox_to_anchor=(1.0, 0.08)); fig.tight_layout(); fig.savefig("figs/fig5.png"); fig.savefig("figs/fig5.pdf"); plt.close(fig)

# Fig 6 - sensitivity heatmaps
fig, axs = plt.subplots(1, 3, figsize=(7.4, 2.9), dpi=200)
for ax, key, cm, ttl, vmin, vmax in ((axs[0], "k", "Blues", "|M′|", 0, 14), (axs[1], "mscope", "Reds", "AUC (M′-scope)", .5, 1), (axs[2], "global", "Greens", "AUC (global)", .5, 1)):
    Z = np.array(HEAT[key]); im = ax.imshow(Z, origin="lower", cmap=cm, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(7)); ax.set_xticklabels(GRID_U, fontsize=6); ax.set_yticks(range(6)); ax.set_yticklabels(GRID_D, fontsize=6)
    ax.set_title(ttl, fontsize=7); ax.set_xlabel("θ_U", fontsize=7)
    for i in range(6):
        for j in range(7): ax.text(j, i, f"{Z[i, j]:.1f}" if key == "k" else f"{Z[i, j]:.2f}", ha="center", va="center", fontsize=4.5)
    fig.colorbar(im, ax=ax, fraction=.05, pad=.04).ax.tick_params(labelsize=6)
axs[0].set_ylabel("θ_Δ", fontsize=7); fig.tight_layout(); fig.savefig("figs/fig6.png"); fig.savefig("figs/fig6.pdf"); plt.close(fig)

# Fig 7 - noise
fig, axs = plt.subplots(1, 2, figsize=(7.2, 2.9), dpi=200); nz = (0, .1, .2, .3)
for m, lab, mk, ls, c in (("mscope", "M′-scope thresholds", "o", "-", COL["mscope"]), ("global", "Global thresholds (default)", "s", "-", COL["global_"]),
                          ("noreduc_mean", "No reduction, mean thresholds", "^", "--", COL["noreduc_mean"]), ("noreduc_fixed", "No reduction, fixed (0.5,0.5,0.5)", "D", "-", COL["noreduc_fixed"]), ("crisp", "Crisp, T ≥ 0.5", "v", "-", COL["crisp"])):
    for ax, idx in ((axs[0], 0), (axs[1], 1)):
        ax.plot(nz, [S["T13"][m][str(z)][idx] for z in nz], marker=mk, ls=ls, color=c, ms=4, lw=1.3, label=lab)
axs[0].set_ylabel("Mean CDI at δ = 0 (false drift)"); axs[1].set_ylabel("AUC, δ = 0 vs δ = 0.2")
for ax in axs: ax.set_xlabel("Cell noise rate"); ax.grid(alpha=.3)
axs[0].legend(fontsize=5.5, frameon=False, loc="lower right"); fig.tight_layout(); fig.savefig("figs/fig7.png"); fig.savefig("figs/fig7.pdf"); plt.close(fig)

# Fig 8 - concepts and time against the number of attributes
fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.0), dpi=200); na_ = (20, 40, 80)
for m, lab, mk, ls, c in (("crisp", "Crisp, no reduction", "v", "-", COL["crisp"]), ("crisp_scr", "Crisp + U,Δ reduction", "v", "--", COL["crisp"]),
                          ("noreduc_fixed", "Fixed thr., no reduction", "D", "-", COL["noreduc_fixed"]), ("fixed_scr", "Fixed thr. + U,Δ reduction", "D", "--", COL["noreduc_fixed"]),
                          ("noreduc_mean", "Mean thr., no reduction", "^", "-", COL["noreduc_mean"]), ("global", "Global mean thr. + reduction (default)", "o", "--", COL["mscope"])):
    axs[0].plot(na_, [S["T14"][f"{n}|{m}"]["nc"] for n in na_], marker=mk, ls=ls, color=c, ms=4, lw=1.4, label=lab)
    axs[1].plot(na_, [S["T14"][f"{n}|{m}"]["sec"] for n in na_], marker=mk, ls=ls, color=c, ms=4, lw=1.4)
axs[0].axhline(CAP, ls=":", c="k", lw=.8); axs[0].text(21, CAP * 1.08, "cap = 1500 (intractable)", fontsize=5.5)
axs[0].set_yscale("log"); axs[1].set_yscale("log"); axs[0].set_ylim(15, 2600)
for ax in axs: ax.set_xticks(na_); ax.set_xlabel("Number of attributes"); ax.grid(alpha=.3)
axs[0].set_ylabel("Mean number of concepts"); axs[1].set_ylabel("Concept generation time (s)"); axs[0].legend(fontsize=5.5, frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig("figs/fig8.png"); fig.savefig("figs/fig8.pdf"); plt.close(fig)

# Fig 9 - scenarios, each measure scaled by its own maximum
fig, ax = plt.subplots(figsize=(6.4, 3.2), dpi=200); w = .2; sc = ("none", "shift0.10", "shift0.20", "structural")
for i, (key, lab, c) in enumerate((("global", "CDI, global thresholds (default)", COL["mscope"]), ("crisp", "CDI, crisp T ≥ 0.5", COL["crisp"]), ("raw", "Raw (T,I,F) distance", COL["raw"]), ("ham", "Hamming, default binary tables", COL["global_"]))):
    v = np.array([S["T16"][key]["means"][s] for s in sc]); ax.bar(np.arange(4) + (i - 1.5) * w, v / v.max(), w, label=lab, color=c)
ax.set_xticks(range(4)); ax.set_xticklabels(["No change", "Calibration\nshift 0.10", "Calibration\nshift 0.20", "Structural\ndrift δ = 0.2"], fontsize=7)
ax.set_ylabel("Mean value, scaled by method maximum"); ax.set_ylim(0, 1.25); ax.grid(axis="y", alpha=.3); ax.legend(fontsize=6.5, frameon=False, ncol=2, loc="upper left")
fig.tight_layout(); fig.savefig("figs/fig9.png"); fig.savefig("figs/fig9.pdf"); plt.close(fig)
print("figures saved in the folder 'figs'; Shannon-kept attributes that were uninformative:", S["shannon_kept_uninformative_share"])